# What is self-attention: static and contextual embeddings
1. One-hot and bag of words for two tiny sentences.
2. Static word embeddings learned from 116,830 real English sentences (the English side of the
   Keras English-French dataset), by counting which words appear near each other (PPMI + SVD).
3. A static embedding gives "bank" the same vector in every sentence.
4. A simple self-attention, used here as a black box (it is built step by step in the next Note),
   gives "bank" a different vector in "money bank grows" and in "river bank flows".

In [1]:
import collections
import re
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.linalg import svds
import keras

np.set_printoptions(precision=3, suppress=True)

I0000 00:00:1791073618.532853 1586586 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791073618.547499 1586586 cuda_fft.cc:470] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
I0000 00:00:1791073618.559042 1586586 cuda_dnn.cc:6964] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
I0000 00:00:1791073618.563025 1586586 cuda_blas.cc:1416] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1791073618.569773 1586586 collection_registry.cc:88] Trying to register 2 metrics with the same name: /coordination_service/agent/enabled. The old value will be erased in order to 

## 1. One-hot encoding and bag of words

In [2]:
sentences = ["mat cat mat", "cat rat rat"]
vocab_small = ["mat", "cat", "rat"]
onehot = {w: np.eye(3, dtype=int)[i] for i, w in enumerate(vocab_small)}
for s in sentences:
    print(s, "| one-hot:", [onehot[w].tolist() for w in s.split()],
          "| bag of words:", [s.split().count(w) for w in vocab_small])

mat cat mat | one-hot: [[1, 0, 0], [0, 1, 0], [1, 0, 0]] | bag of words: [2, 1, 0]
cat rat rat | one-hot: [[0, 1, 0], [0, 0, 1], [0, 0, 1]] | bag of words: [0, 1, 2]


## 2. Average meaning, the worked example
Suppose each use of "apple" as a fruit pulls its vector towards (taste 1, technology 0), and each use
as a company towards (0, 1). The vector reflects the mix of the whole corpus.

In [3]:
fruit, company = np.array([1.0, 0.0]), np.array([0.0, 1.0])
for n_fruit, n_company in ((9000, 1000), (1000, 9000)):
    print(n_fruit, "fruit,", n_company, "company ->", (n_fruit * fruit + n_company * company) / (n_fruit + n_company))

9000 fruit, 1000 company -> [0.9 0.1]
1000 fruit, 9000 company -> [0.1 0.9]


## 3. Static embeddings from real text
Corpus: the 116,830 distinct English sentences of the English-French dataset.
Count how often each pair of words appears within 4 words of each other, weight the counts by
positive pointwise mutual information (PPMI), and keep the 50 strongest SVD directions.

In [4]:
path = keras.utils.get_file("fra-eng.zip", origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
                            extract=True)
lines = sorted(set(l.split("\t")[0] for l in open(f"{path}/fra.txt", encoding="utf-8")))
sents = [re.findall(r"[a-z']+", l.lower()) for l in lines]
count = collections.Counter(w for s in sents for w in s)
vocab = [w for w, c in count.items() if c >= 5]
idx = {w: i for i, w in enumerate(vocab)}
print(len(lines), "sentences,", len(vocab), "words used at least 5 times")
print("sentences with 'bank':", sum("bank" in s for s in sents),
      " with 'bank' and 'river':", sum("bank" in s and "river" in s for s in sents))

116830 sentences, 5287 words used at least 5 times
sentences with 'bank': 95  with 'bank' and 'river': 0


In [5]:
rows, cols = [], []
for s in sents:
    ids = [idx[w] for w in s if w in idx]
    for a in range(len(ids)):
        for b in range(max(0, a - 4), min(len(ids), a + 5)):
            if a != b:
                rows.append(ids[a]); cols.append(ids[b])
C = sp.coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(vocab),) * 2).tocsr()
total = C.sum()
p_word = np.asarray(C.sum(1)).ravel() / total
p_ctx = (np.asarray(C.sum(0)).ravel() / total) ** 0.75
p_ctx /= p_ctx.sum()
C = C.tocoo()
pmi = np.log((C.data / total) / (p_word[C.row] * p_ctx[C.col]))
keep = pmi > 0
PPMI = sp.csr_matrix((pmi[keep], (C.row[keep], C.col[keep])), shape=C.shape)
U, S, _ = svds(PPMI, k=50, random_state=0)
E = U * np.sqrt(S)
E /= np.linalg.norm(E, axis=1, keepdims=True)        # unit length: dot product = cosine similarity
print("embedding matrix:", E.shape)


def neighbours(w, k=6):
    sims = E @ E[idx[w]]
    return [vocab[i] for i in np.argsort(-sims)[1:k + 1]]


for w in ["bank", "money", "river", "apple", "phone"]:
    print(f"{w:6s} ->", neighbours(w))
cos = lambda a, b: float(a @ b / np.linalg.norm(a) / np.linalg.norm(b))
print("static bank: similarity to money %.2f, to river %.2f" % (cos(E[idx["bank"]], E[idx["money"]]), cos(E[idx["bank"]], E[idx["river"]])))

embedding matrix: (5287, 50)
bank   -> ['shop', 'bookstore', 'supermarket', 'hall', 'convenience', 'hospital']
money  -> ['information', 'more', 'cash', 'stamps', 'earn', 'need']
river  -> ['lake', 'sea', 'across', 'flows', 'village', 'bridge']
apple  -> ['pie', 'fruit', 'apples', 'baked', 'banana', 'orange']
phone  -> ['telephone', 'cell', 'passport', 'address', 'number', 'card']
static bank: similarity to money 0.40, to river 0.24


## 4. A static embedding is the same in every sentence
Keras' `Embedding` layer with these vectors as its weights: a lookup table.

In [6]:
emb = keras.layers.Embedding(len(vocab), 50, weights=[E.astype("float32")], trainable=False)
s1, s2 = "money bank grows".split(), "river bank flows".split()
out = emb(np.array([[idx[w] for w in s1], [idx[w] for w in s2]])).numpy()
print("bank in sentence 1 == bank in sentence 2:", np.array_equal(out[0, 1], out[1, 1]))

W0000 00:00:1791073624.254477 1586586 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1791073624.256666 1586586 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1186 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


bank in sentence 1 == bank in sentence 2: True


## 5. Self-attention as a black box: static in, contextual out
The simplest self-attention (built in the next Note): each output vector is a weighted average of
all the input vectors of the sentence, with weights from their similarity.

In [7]:
def self_attention(X):
    scores = X @ X.T
    W = np.exp(scores - scores.max(1, keepdims=True))
    W /= W.sum(1, keepdims=True)
    return W @ X


rows = []
for s in (s1, s2):
    X = np.array([E[idx[w]] for w in s])
    Y = self_attention(X)
    y_bank = Y[s.index("bank")]
    rows.append(dict(sentence=" ".join(s), sim_money=cos(y_bank, E[idx["money"]]), sim_river=cos(y_bank, E[idx["river"]]),
                     sim_static_bank=cos(y_bank, E[idx["bank"]])))
rows.insert(0, dict(sentence="static (any sentence)", sim_money=cos(E[idx["bank"]], E[idx["money"]]),
                    sim_river=cos(E[idx["bank"]], E[idx["river"]]), sim_static_bank=1.0))
sims = pd.DataFrame(rows).round(2)
sims.to_csv("data/bank_similarity.csv", index=False)
sims

,sentence,sim_money,sim_river,sim_static_bank
0,static (any sentence),0.40,0.24,1.00
1,money bank grows,0.68,0.13,0.90
2,river bank flows,0.21,0.71,0.82


## 6. A 2-D map of the vectors (for the figure)
PCA of a few related words, plus the contextual vectors of "bank" from the two sentences.

In [8]:
words = ["money", "cash", "loan", "account", "grows", "river", "lake", "boat", "flows", "bridge", "bank"]
pts = np.array([E[idx[w]] for w in words])
ctx = []
for s in (s1, s2):
    X = np.array([E[idx[w]] for w in s])
    ctx.append(self_attention(X)[1])
allp = np.vstack([pts, ctx])
centre = allp.mean(0)
_, _, Vt = np.linalg.svd(allp - centre, full_matrices=False)
xy = (allp - centre) @ Vt[:2].T
labels = words + ["bank in 'money bank grows'", "bank in 'river bank flows'"]
kinds = ["money"] * 5 + ["river"] * 5 + ["static"] + ["contextual"] * 2
m = pd.DataFrame(dict(label=labels, kind=kinds, x=xy[:, 0], y=xy[:, 1])).round(3)
m.to_csv("data/bank_map.csv", index=False)
m

,label,kind,x,y
0,money,money,-0.604,0.078
1,cash,money,-0.708,0.098
2,loan,money,-0.609,0.058
3,account,money,-0.378,0.143
4,grows,money,-0.038,-1.048
5,river,river,0.695,0.121
6,lake,river,0.595,-0.138
7,boat,river,0.457,0.150
8,flows,river,0.605,0.239
9,bridge,river,0.597,-0.034
